In [ ]:
import requests
import time
import os 
import pandas as pd
import codecs
import re
import unicodedata
from pathlib import Path
from rapidfuzz import process, fuzz


from path import *


In [ ]:
df = pd.read_csv(Twitch_PATH, encoding="utf-8", encoding_errors="replace")

In [ ]:
def cp1252_fallback(exc):
    if isinstance(exc, UnicodeDecodeError):
        # If utf-8 fails, try decoding the bad byte with cp1252
        res = exc.object[exc.start:exc.end].decode("cp1252")
        return (res, exc.end)
    raise TypeError(f"cannot handle {exc}")

# Register our custom error handler
codecs.register_error("cp1252_fallback", cp1252_fallback)

# Load the file using the fallback
twitch = pd.read_csv(Twitch_PATH, encoding="utf-8", encoding_errors="cp1252_fallback")

# Fix R-style unicode escapes (e.g. <U+00E9> for é)
twitch["Game"] = twitch["Game"].str.replace(r"<U\+([0-9A-Fa-f]{4})>", lambda m: chr(int(m.group(1), 16)), regex=True)

In [ ]:
df


In [ ]:
steam = pd.read_csv(Steam_PATH, encoding="utf-8", encoding_errors="cp1252_fallback")
steam

In [ ]:
# imports for the matching
import unicodedata
from rapidfuzz import process, fuzz

# roman numerals -> numbers, so "Final Fantasy VII" and "Final Fantasy 7" look the same
roman = {"ii": "2", "iii": "3", "iv": "4", "v": "5", "vi": "6", "vii": "7", "viii": "8", "ix": "9",
         "xi": "11", "xii": "12", "xiii": "13", "xiv": "14", "xv": "15", "xvi": "16"}

def clean_name(name):
    # makes game names easier to compare
    if pd.isna(name):
        return ""
    name = str(name).lower()
    name = re.sub(r"[™®©]", " ", name)                  # remove TM, R, C symbols
    name = unicodedata.normalize("NFKD", name)
    name = "".join(c for c in name if not unicodedata.combining(c))   # remove accents
    name = name.replace("&", " and ")
    name = re.sub(r"[^\w\s]", " ", name)                # remove punctuation
    name = re.sub(r"\s+", " ", name).strip()
    name = re.sub(r"^the ", "", name)                   # remove "the" at the start
    words = name.split()
    words = [roman.get(w, w) for w in words]
    return " ".join(words)

def get_numbers(name):
    # all numbers in a name, so "madden nfl 20" is not matched with "madden nfl 22"
    return re.findall(r"\d+", name)


In [ ]:
# steam: drop columns that we don't need
drop_cols = ["detailed_description", "about_the_game", "short_description", "reviews", "header_image",
             "support_url", "support_email", "notes", "packages", "screenshots", "movies"]
steam_small = steam.drop(columns=drop_cols)

# add clean name and release date as a real date
steam_small["clean_name"] = steam_small["name"].apply(clean_name)
steam_small["release"] = pd.to_datetime(steam_small["release_date"], errors="coerce")

# some games are in steam more than once -> keep the earliest release date
steam_small = steam_small[steam_small["clean_name"] != ""]
steam_small = steam_small.sort_values("release")
steam_small = steam_small.drop_duplicates(subset="clean_name", keep="first")
steam_small = steam_small.drop(columns="release")
print(steam_small.shape)


In [ ]:
# twitch: clean name
twitch["clean_name"] = twitch["Game"].apply(clean_name)

steam_names = steam_small["clean_name"].tolist()
steam_names_set = set(steam_names)

# for every twitch game find the steam name that fits (exact first, then fuzzy)
matches = {}    # twitch clean name -> (steam clean name, score)

for game in twitch["clean_name"].unique():
    if game == "":
        continue

    if game in steam_names_set:
        matches[game] = (game, 100)
        continue

    # fuzzy: look at the 5 best candidates
    best = process.extract(game, steam_names, scorer=fuzz.ratio, score_cutoff=93, limit=5)
    for steam_name, score, index in best:
        same_numbers = get_numbers(game) == get_numbers(steam_name)
        # short names need a higher score, otherwise "dreams" would match "dream"
        if len(game) < 12 and score < 96:
            continue
        if same_numbers:
            matches[game] = (steam_name, score)
            break

print("exact:", sum(1 for m in matches.values() if m[1] == 100))
print("fuzzy:", sum(1 for m in matches.values() if m[1] < 100))


In [ ]:
# add the matched steam name and score to twitch
twitch["steam_clean_name"] = twitch["clean_name"].apply(lambda x: matches[x][0] if x in matches else None)
twitch["steam_match_score"] = twitch["clean_name"].apply(lambda x: matches[x][1] if x in matches else None)

# left join: all twitch rows stay, steam columns are empty if there is no match
steam_small = steam_small.rename(columns={"clean_name": "steam_clean_name"})
df = twitch.merge(steam_small, how="left", on="steam_clean_name")
df = df.drop(columns=["clean_name", "steam_clean_name"])
df.shape


In [ ]:
# twitch games that are not in steam -> not_in_steam.csv (one row per game)
not_in_steam = df[df["appid"].isna()]
not_in_steam = not_in_steam.groupby("Game").agg(months=("Month", "count"),
                                                total_hours_watched=("Hours_watched", "sum"))
not_in_steam = not_in_steam.sort_values("total_hours_watched", ascending=False).reset_index()
not_in_steam.to_csv(Not_in_steam_PATH, index=False)
print(len(not_in_steam), "games not in steam")
not_in_steam.head()


In [ ]:
# keep only the rows that matched a steam game
df = df[df["appid"].notna()].reset_index(drop=True)
df.to_csv(COMBINED_Twitch_Steam_PATH, index=False)
print(df.shape, df["Game"].nunique(), "games")
df.head()
